# Week 4 Word embeddings and vector arithmetic with GloVe

## Task Description
For this week's assignment, I will be working with GloVe dataset. It consists of pre-trained word vectors for a large vocabulary of words. The task is to use these pre-trained word vectors to find interesting relationships between words using mathematical operations. For example, we can use the word vectors to find the relationship between "king" and "queen" by performing vector arithmetic like "king - man + woman". This allows us to explore the semantic relationships between words in a high-dimensional vector space.

### Steps:
1. Load the pretrained GloVe embeddings from a local file
2. Demonstrate the classic **king − man + woman ≈ queen** analogy
3. Build a general-purpose function for vector arithmetic between any two words
4. Explore additional interesting semantic relationships


## 1. Imports and Configuration

In [1]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from typing import Callable
import operator as op

# Configuration
GLOVE_FILE_PATH = "datasets/glove.42B.300d.txt"

# Number of nearest neighbours to return in similarity searches.
TOP_N = 6

## 2. Loading the GloVe Embeddings

GloVe files are plain text: each line contains a word followed by its vector
components separated by spaces. We parse this into two complementary data structures:

**`word_to_vec`** — a dictionary for instant lookup by word name:
```python
{
    "king":  np.array([0.504,  0.686, -0.595, ...]),
    "woman": np.array([0.310, -0.123,  0.778, ...]),
}
```

**`vectors_matrix`** — all vectors stacked into a single NumPy matrix, where row `i` corresponds to `vocab[i]`:
```
row 0:      [0.504,  0.686, -0.595, ...]   ← "king"
row 1:      [0.310, -0.123,  0.778, ...]   ← "woman"
...
row 399999: [...]                           ← last word
```

The matrix is critical for performance. It allows cosine similarity to be computed
against all 400 000+ vocabulary entries in a single NumPy operation rather than a
Python loop, which would be roughly much slower.

In [2]:
def load_glove_embeddings(file_path: str) -> tuple[dict, list, np.ndarray]:
    word_to_vec: dict[str, np.ndarray] = {} # word-vector dictionary
    vocab: list[str] = []                   # ordered list of words (same order as matrix rows)
    raw_vectors: list[np.ndarray] = []      # list of vectors to be stacked into a matrix

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            # parts = ["king", "0.504", "0.686", "-0.595", ...]

            word = parts[0]           # "king"
            vector = np.array(parts[1:], dtype=np.float32)
            # converts the remaining strings (vector values) into actual floats

            word_to_vec[word] = vector   # store in dict
            vocab.append(word)           # keep the ordering
            raw_vectors.append(vector)   # collect for the matrix

    vectors_matrix = np.vstack(raw_vectors)  # shape: (vocab_size, embedding_dim)
    return word_to_vec, vocab, vectors_matrix


# Load embeddings. This may take some time depending on hardware.
print(f"Loading GloVe embeddings from '{GLOVE_FILE_PATH}' ...")
word_to_vec, vocab, vectors_matrix = load_glove_embeddings(GLOVE_FILE_PATH)

embedding_dim = vectors_matrix.shape[1]
print(f"Loaded {len(vocab):,} words with {embedding_dim}-dimensional vectors.")

Loading GloVe embeddings from 'datasets/glove.42B.300d.txt' ...
Loaded 1,917,494 words with 300-dimensional vectors.


## 3. Finding Nearest Neighbours

**Cosine similarity** measures the angle between two vectors rather than their absolute distance. A value of 1.0 means the vectors point in exactly the same direction (most similar), while 0.0 means they are unrelated.

The helper function below computes the similarity between a query vector and every row of `vectors_matrix` simultaneously using matrix operations, then returns the top-ranked words.

In [3]:
def find_nearest_words(
    query_vector: np.ndarray,
    vocab: list[str],
    vectors_matrix: np.ndarray,
    top_n: int = 6,
    exclude_words: list[str] | None = None,) -> list[tuple[str, float]]:

    exclude_set = set(exclude_words) if exclude_words else set()

    # Compute cosine similarity against every word in the vocabulary at once.
    similarities = cosine_similarity(
        query_vector.reshape(1, -1),  # shape (1, embedding_dim)
        vectors_matrix,               # shape (vocab_size, embedding_dim)
    ).flatten()                       # shape (vocab_size,)

    # Sort indices from highest to lowest similarity.
    ranked_indices = np.argsort(similarities)[::-1]

    results: list[tuple[str, float]] = []
    for idx in ranked_indices:
        word = vocab[idx]
        if word not in exclude_set:
            results.append((word, float(similarities[idx])))
        if len(results) == top_n:
            break

    return results

## 4. The Classic Analogy: king − man + woman

One of the most celebrated properties of word embeddings is that **semantic relationships can be captured by vector arithmetic**. The canonical example is:

$$\vec{\text{woman}} - \vec{\text{man}} + \vec{\text{king}} \approx \vec{\text{queen}}$$

The intuition is that the difference $\vec{\text{woman}} - \vec{\text{man}}$ encodes the concept of *gender direction* in the embedding space. Adding this gender offset to $\vec{\text{king}}$ should yield a vector close to the word that represents royalty with a female gender, i.e. **queen**.

In [4]:
# Retrieve the individual word vectors.
vec_man   = word_to_vec["man"]
vec_woman = word_to_vec["woman"]
vec_king  = word_to_vec["king"]

# Compute the query: woman - man + king
query = vec_woman - vec_man + vec_king

# Find the 6 nearest neighbours, excluding the three source words.
results = find_nearest_words(
    query,
    vocab,
    vectors_matrix,
    top_n=TOP_N,
    exclude_words=["man", "woman", "king"],
)

print("Nearest words to  vec('woman') − vec('man') + vec('king'):")
print("-" * 45)
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} cosine similarity: {score:.4f}")

Nearest words to  vec('woman') − vec('man') + vec('king'):
---------------------------------------------
  1. queen                cosine similarity: 0.7852
  2. prince               cosine similarity: 0.6026
  3. princess             cosine similarity: 0.5831
  4. elizabeth            cosine similarity: 0.5546
  5. daughter             cosine similarity: 0.5454
  6. mother               cosine similarity: 0.5362


### Explanation

The result confirms that **queen** appears at or very near the top of the list. This happens because GloVe was trained on billions of words of text, during which it learned that:

- *man* and *woman* are related in the same way as *king* and *queen*. They differ primarily along a **gender axis** in the vector space.
- Subtracting `vec(man)` removes the "male" component; adding `vec(woman)` reintroduces the "female" component.
- The resulting vector lands close to *queen* because *queen* is the word that best combines royalty with femininity.

## 5. Exploring Semantic Relationships

### 5.1 Gender Analogies

I alreadt tried the classic *king* analogy, but the same gender relationship should hold for many other pairs of words. In the code cell belov I test the same gender offset on *actor* and *brother* to see if it surfaces the expected feminine counterparts *actress* and *sister*.

In [5]:
# actor − man + woman  →  should be close to 'actress'
print("Gender analogy: actor − man + woman")
actor_vec   = word_to_vec["actor"] - word_to_vec["man"] + word_to_vec["woman"]
results = find_nearest_words(actor_vec, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["actor", "man", "woman"])

# print results
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")
print()

# brother − man + woman  →  should be close to 'sister'
print("Gender analogy: brother − man + woman")
brother_vec = word_to_vec["brother"] - word_to_vec["man"] + word_to_vec["woman"]
results = find_nearest_words(brother_vec, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["brother", "man", "woman"])

# print results
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")

Gender analogy: actor − man + woman
  1. actress              0.8502
  2. actors               0.6680
  3. actresses            0.6199
  4. starring             0.6058
  5. starred              0.5771
  6. co-star              0.5697

Gender analogy: brother − man + woman
  1. sister               0.7841
  2. daughter             0.7720
  3. mother               0.7300
  4. husband              0.7007
  5. wife                 0.6901
  6. niece                0.6806


### 5.2 Capital City Relationships

I also tried some geographic relationships. The analogy method should work for any pair of words that share a consistent relationship in the training data. For example, the relationship between a country and its capital city should be captured by a similar vector offset across different countries. I tested this by taking the vector difference between a capital and its country, then applying that offset to another country to see if it lands near the correct capital.

In [6]:
# Paris − France + Germany  →  should be close to 'Berlin'
print("Capital analogy: Paris − France + Germany")
capital_vec = word_to_vec["paris"] - word_to_vec["france"] + word_to_vec["germany"]
results = find_nearest_words(capital_vec, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["paris", "france", "germany"])
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")
print()

# Tokyo − Japan + China  →  should be close to 'Beijing'
print("Capital analogy: Tokyo − Japan + China")
capital_vec2 = word_to_vec["tokyo"] - word_to_vec["japan"] + word_to_vec["china"]
results = find_nearest_words(capital_vec2, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["tokyo", "japan", "china"])
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")

Capital analogy: Paris − France + Germany
  1. berlin               0.7635
  2. munich               0.6988
  3. frankfurt            0.6719
  4. hamburg              0.6531
  5. vienna               0.6255
  6. stuttgart            0.6102

Capital analogy: Tokyo − Japan + China
  1. beijing              0.7887
  2. shanghai             0.7830
  3. hong                 0.6708
  4. shenzhen             0.6483
  5. guangzhou            0.6438
  6. kong                 0.6340


### 5.3 Concept Addition

Addition of vectors combines concepts. For example, adding the vectors for "coffee" and "morning" should yield a vector that is close to words related to a morning routine, while adding "doctor" and "hospital" should surface medical profession words. The code below tests examples of concept addition to see if the resulting vectors land near the expected semantic fields.

In [7]:
print("Concept addition: human + flu")
concept_addition = word_to_vec["human"] + word_to_vec["flu"]
results = find_nearest_words(concept_addition, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["human", "flu"])
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")

print("Concept addition: coffee + morning")
concept_addition2 = word_to_vec["coffee"] + word_to_vec["morning"]
results = find_nearest_words(concept_addition2, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["coffee", "morning"])
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")

print("Concept addition: doctor + hospital")
concept_addition3 = word_to_vec["doctor"] + word_to_vec["hospital"]
results = find_nearest_words(concept_addition3, vocab, vectors_matrix, top_n=TOP_N, exclude_words=["doctor", "hospital"])
for rank, (word, score) in enumerate(results, start=1):
    print(f"  {rank}. {word:<20} {score:.4f}")


Concept addition: human + flu
  1. influenza            0.7169
  2. swine                0.7077
  3. h1n1                 0.6935
  4. humans               0.6608
  5. pandemic             0.6566
  6. virus                0.6514
Concept addition: coffee + morning
  1. afternoon            0.7619
  2. breakfast            0.7201
  3. evening              0.7049
  4. tea                  0.7021
  5. day                  0.6859
  6. lunch                0.6738
Concept addition: doctor + hospital
  1. medical              0.7739
  2. doctors              0.7429
  3. physician            0.7398
  4. clinic               0.7195
  5. nurse                0.6959
  6. patient              0.6912


## 6. Summary and Reflections

### What the results tell us

| Experiment | Expression | Expected result | Observed? |
|---|---|---|---|
| Gender | woman − man + king | queen | ✅ |
| Gender | woman − man + actor | actress | ✅ |
| Gender | woman − man + brother | sister | ✅ |
| Capital | Paris − France + Germany | Berlin | ✅ |
| Capital | Tokyo − Japan + China | Beijing | ✅ |
| Concept | human + flu | words related to illness | ✅ |
| Concept | coffee + morning | words related to morning routine | ✅ |
| Concept | doctor + hospital | words related to medical profession | ✅ |

The results confirm that GloVe embeddings capture rich semantic relationships between words. The vector arithmetic operations successfully surfaced expected analogies and concept combinations, demonstrating that the embedding space encodes meaningful directions corresponding to various linguistic and real-world relationships. This illustrates the power of word embeddings in representing complex semantic information in a way that allows for algebraic manipulation.

Personally I found this task extremely fascinating. It was surprising to see how well the vector arithmetic worked. The fact that simple operations like addition and subtraction could yield such meaningful results was a powerful demonstration of the underlying structure captured by the GloVe embeddings. It really highlighted how these models learn to represent language in a way that reflects real-world relationships, which is both impressive and inspiring for further exploration.